In [ ]:
# 📅 2026-09-04. OpenAI Prompting & Chat History Management
- **학습 일자:** 2026-09-04
- **주요 학습:** Prompt Structuring, Few-shot Prompting, System Role Control, Multi-turn Chat Management

---

### 📌 목차
1. [⚙️ 공통 환경 설정 및 클라이언트 생성](#setup)
2. [📝 Q1. 감정 분석 프롬프트 및 점수 분류 (Zero-shot)](#q1)
3. [📝 Q2. 예시 기반 감정 분석 점수 분류 (Few-shot)](#q2)
4. [📝 Q3. System Message를 활용한 학습 계획 에이전트](#q3)
5. [📝 Q4. System Message 조건부 제약사항(Guardrails) 설정](#q4)
6. [📝 Q5. Assistant Message를 활용한 In-context Learning](#q5)
7. [📝 Q6. 대화 히스토리 리스트 관리 (History Append)](#q6)
8. [📝 Q7. Multi-turn 대화 루프 구현 (Chat Loop)](#q7)

---

client = OpenAI(api_key="<OPENAI_API_TOKEN>")

# Define a multi-line prompt to classify sentiment
prompt = """distinguish sentences into number 1 to 5 which small number is more positive
for example good is little positive so 2 and bad is little negative so 4 :
1. Unbelievably good!
2. Shoes fell apart on the second use.
3. The shoes look nice, but they aren't very comfortable.
4. Can't wait to show them off!"""

# Create a request to the Chat Completions endpoint
response = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[{"role": "user", "content": prompt}],
  max_completion_tokens=100
)

print(response.choices[0].message.content)
---

client = OpenAI(api_key="<OPENAI_API_TOKEN>")

# Add the final example
prompt = """Classify sentiment as 1-5 (negative to positive):
1. ____
2. Love these! = 5
3. Unbelievably good! =
4. Shoes fell apart on the second use. =
5. The shoes look nice, but they aren't very comfortable. =
6. Can't wait to show them off! = """

response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}], max_completion_tokens=100)
print(response.choices[0].message.content)

---
client = OpenAI(api_key="<OPENAI_API_TOKEN>")

# Create a request to the Chat Completions endpoint
response = client.chat.completions.create(
  model="gpt-4o-mini",
  max_completion_tokens=150,
  messages=[
    {"role": "system",
     "content": "You are a study planning assistant that creates plans for learning new skills."},
    {"role": "user",
     "content": "I want to learn to speak Dutch."}
  ]
)

# Extract the assistant's text response
print(response.choices[0].message.content)

---
client = OpenAI(api_key="<OPENAI_API_TOKEN>")

sys_msg = """You are a study planning assistant that creates plans for learning new skills.

If these skills are non related to languages, return the message:

'Apologies, to focus on languages, we no longer create learning plans on other topics.'
"""

# Create a request to the Chat Completions endpoint
response = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {"role": "system", "content": sys_msg},
    {"role": "user", "content": "Help me learn to roller."}
  ]
)

print(response.choices[0].message.content)
---

client = OpenAI(api_key="<OPENAI_API_TOKEN>")

response = client.chat.completions.create(
    model="gpt-4o-mini",
    # Add a user and assistant message for in-context learning
    messages=[
        {"role": "system", "content": "You are a helpful Geography tutor that generates concise summaries for different countries."},
        {"role" : "user" , "content": "Give me a quick summary of Portugal."},
        {"role" :"assistant", "content": "Portugal is a country in Europe that borders Spain. The capital city is Lisboa."},
        {"role": "user", "content": "Give me a quick summary of Greece."}
    ]
)

print(response.choices[0].message.content)
---

client = OpenAI(api_key="<OPENAI_API_TOKEN>")

messages = [
    {"role": "system", "content": "You are a helpful math tutor that speaks concisely."},
    {"role": "user", "content": "Explain what pi is."}
]

# Send the chat messages to the model
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    max_completion_tokens=100
)

# Extract the assistant message from the response
assistant_dict = {"role": "assistant", "content": "pi is 3.14.."}

# Add assistant_dict to the messages dictionary
messages.append(assistant_dict)
print(messages)
---

client = OpenAI(api_key="<OPENAI_API_TOKEN>")

messages = [{"role": "system", "content": "You are a helpful math tutor that speaks concisely."}]
user_msgs = ["Explain what pi is.", "Summarize this in two bullet points."]

# Loop over the user questions
for q in user_msgs:
    print("User: ", q)

    # Create a dictionary for the user message from q and append to messages
    user_dict = {"role": "user", "content": q}
    messages.append(user_dict)

    # Create the API request
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages = messages,
        max_completion_tokens=100
    )

    # Append the assistant's message to messages
    assistant_dict = {"role": "assistant", "content": response.choices[0].message.content}
    messages.append(assistant_dict)
    print("Assistant: ", response.choices[0].message.content, "\n")